# 01b FabricNET Intertwined Frame Vector (IFV) Sliding Window Regression
## Experiment ID: `EXP-FABRICNET-IFV-SLIDING-WINDOW`

### 1. Mathematical Objective & Literature Prior
This notebook implements the published Intertwined Frame Vector (IFV) feature extraction method for woven fabric density estimation according to Secer & Ercan (2022/2023, MDPI Sustainability 15(21), 15197, `doi:10.3390/su152115197`).

### Mathematical Formulation of Nested Concentric Frames:
Across each image ($480 \times 640$ pixels), a sliding window of size $160 \times 160$ pixels moves with a stride of 80 pixels:
- Grid: $Y = \{0, 80, 160, 240, 320\}$ (5 steps), $X = \{0, 80, 160, 240, 320, 400, 480\}$ (7 steps) $\to$ 35 sliding windows per specimen.
- For each $160 \times 160$ window, 4 nested concentric rectangular frames $k \in \{0, 1, 2, 3\}$ of thickness 20 pixels are extracted:
  - Frame 0: Outer perimeter $[0:160, 0:160]$ excluding $[20:140, 20:140]$.
  - Frame 1: Annulus $[20:140, 20:140]$ excluding $[40:120, 40:120]$.
  - Frame 2: Annulus $[40:120, 40:120]$ excluding $[60:100, 60:100]$.
  - Frame 3: Central core $[60:100, 60:100]$.
- For each frame $k$, compute:
  1. Intensity-weighted centroid $(\bar{x}_k, \bar{y}_k)$ and offset from window center $(80, 80)$: $dx_k, dy_k, dr_k = \sqrt{dx_k^2 + dy_k^2}$.
  2. Frame mean intensity $\mu_k$ and standard deviation $\sigma_k$.
  3. Frame 1D directional projection variances: $\mathrm{Var}(\mathrm{Proj}_h), \mathrm{Var}(\mathrm{Proj}_v)$.
  - Yields $4 \times 7 = 28$ features per window.
- Across the 35 windows of the specimen, compute window-level mean and standard deviation: $28 \times 2 = 56$ features.

### Invariants:
1. **Identical Disjoint Splitting**: Utilizes the exact partition split from `experiments/learned_models/results/fabricnet_splits.json` (78 train / 26 val / 26 test).
2. **Deterministic Seed**: Locked to `seed=42`.
3. **Model Paradigms**: Random Forest, XGBoost, and MLP Regressor.

In [1]:
# The mechanistic justification for this step is: initialize scientific computing libraries, ensure reproducible pseudo-random states, and verify hardware execution context.
import os
import glob
import json
import time
import numpy as np
import pandas as pd
import cv2
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))


CUDA Available: True
Device Name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
# The mechanistic justification for this step is: load the exact specimen-disjoint partition splits to ensure rigorous side-by-side comparability with the global texture baseline.
params_path = "data/fabricnet/FabricNet_parameters.xlsx"
df_meta = pd.read_excel(params_path)
print("Metadata shape:", df_meta.shape)

with open("experiments/learned_models/results/fabricnet_splits.json") as f:
    splits = json.load(f)

train_ids = splits['train_ids']
val_ids = splits['val_ids']
test_ids = splits['test_ids']

print(f"Train IDs: {len(train_ids)} | Val IDs: {len(val_ids)} | Test IDs: {len(test_ids)}")


Metadata shape: (130, 5)
Train IDs: 78 | Val IDs: 26 | Test IDs: 26


In [3]:
# The mechanistic justification for this step is: extract the published nested concentric rectangular frame vectors (IFV) across a 160x160 sliding window with stride 80 to measure local yarn periodicities.
def extract_ifv_specimen(img):
    h, w = img.shape
    win_size = 160
    stride = 80
    y_starts = list(range(0, h - win_size + 1, stride))
    x_starts = list(range(0, w - win_size + 1, stride))
    
    masks = []
    cy, cx = win_size / 2.0, win_size / 2.0
    for k in range(4):
        t = k * 20
        m = np.zeros((win_size, win_size), dtype=bool)
        m[t:win_size-t, t:win_size-t] = True
        inner_t = (k + 1) * 20
        if inner_t < win_size / 2.0:
            m[inner_t:win_size-inner_t, inner_t:win_size-inner_t] = False
        masks.append((m, np.where(m)))

    win_feats = []
    for ys in y_starts:
        for xs in x_starts:
            win = img[ys:ys+win_size, xs:xs+win_size]
            wf = []
            for k, (mask, (m_y, m_x)) in enumerate(masks):
                weights = win[m_y, m_x].astype(np.float32)
                total_int = float(np.sum(weights)) + 1e-6
                cy_k = float(np.sum(m_y * weights)) / total_int
                cx_k = float(np.sum(m_x * weights)) / total_int
                dy = cy_k - cy
                dx = cx_k - cx
                dr = float(np.sqrt(dx**2 + dy**2))
                mean_val = float(np.mean(weights))
                std_val = float(np.std(weights))
                
                frame_img = np.zeros((win_size, win_size), dtype=np.float32)
                frame_img[m_y, m_x] = weights
                h_proj = np.sum(frame_img, axis=1) / (np.sum(mask, axis=1) + 1e-6)
                v_proj = np.sum(frame_img, axis=0) / (np.sum(mask, axis=0) + 1e-6)
                var_h = float(np.var(h_proj))
                var_v = float(np.var(v_proj))
                wf.extend([dx, dy, dr, mean_val, std_val, var_h, var_v])
            win_feats.append(wf)
            
    win_arr = np.array(win_feats, dtype=np.float32)
    mean_vec = np.mean(win_arr, axis=0)
    std_vec = np.std(win_arr, axis=0)
    return np.concatenate([mean_vec, std_vec])

features_cache = "experiments/learned_models/results/fabricnet_ifv_features.npz"
if os.path.exists(features_cache):
    cached = np.load(features_cache)
    X_all = cached['X']
    ids_all = cached['ids']
    print(f"Loaded {len(ids_all)} cached IFV feature vectors from {features_cache}.")
else:
    print("Extracting sliding-window IFV features for all 130 specimens...")
    t0 = time.time()
    feature_list = []
    id_list = []
    for idx, row in df_meta.iterrows():
        img_id = int(row['Image id'])
        img_path = f"data/fabricnet/W{img_id:03d}.jpg"
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        feat = extract_ifv_specimen(img)
        feature_list.append(feat)
        id_list.append(img_id)
    X_all = np.array(feature_list, dtype=np.float32)
    ids_all = np.array(id_list)
    np.savez_compressed(features_cache, X=X_all, ids=ids_all)
    print(f"Extracted {len(ids_all)} IFV vectors in {time.time() - t0:.2f}s. Saved to {features_cache}.")

print(f"IFV Feature matrix shape: {X_all.shape}")


Extracting sliding-window IFV features for all 130 specimens...


Extracted 130 IFV vectors in 4.12s. Saved to experiments/learned_models/results/fabricnet_ifv_features.npz.
IFV Feature matrix shape: (130, 56)


In [4]:
# The mechanistic justification for this step is: partition the IFV feature matrices into disjoint train/val/test sets and scale features using exclusively training set statistics.
id_to_idx = {int(img_id): i for i, img_id in enumerate(ids_all)}

train_indices = [id_to_idx[i] for i in train_ids]
val_indices = [id_to_idx[i] for i in val_ids]
test_indices = [id_to_idx[i] for i in test_ids]

X_train = X_all[train_indices]
X_val = X_all[val_indices]
X_test = X_all[test_indices]

meta_by_id = df_meta.set_index('Image id')

y_train_warp = meta_by_id.loc[train_ids, 'Warp'].values.astype(np.float32)
y_train_weft = meta_by_id.loc[train_ids, 'Weft'].values.astype(np.float32)

y_val_warp = meta_by_id.loc[val_ids, 'Warp'].values.astype(np.float32)
y_val_weft = meta_by_id.loc[val_ids, 'Weft'].values.astype(np.float32)

y_test_warp = meta_by_id.loc[test_ids, 'Warp'].values.astype(np.float32)
y_test_weft = meta_by_id.loc[test_ids, 'Weft'].values.astype(np.float32)
test_textures = meta_by_id.loc[test_ids, 'Texture'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_val shape:", X_val_scaled.shape)
print("X_test shape:", X_test_scaled.shape)


X_train shape: (78, 56)
X_val shape: (26, 56)
X_test shape: (26, 56)


In [5]:
# The mechanistic justification for this step is: train XGBoost, Random Forest, and MLP regressors on sliding-window IFV representations and compute comprehensive industrial accuracy metrics.
def bootstrap_mae_ci(y_true, y_pred, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    maes = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        maes.append(float(np.mean(np.abs(y_true[idx] - y_pred[idx]))))
    low = float(np.percentile(maes, (1.0 - ci) / 2.0 * 100.0))
    high = float(np.percentile(maes, (1.0 + ci) / 2.0 * 100.0))
    return [low, high]

def compute_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    pct_within_1 = float(np.mean(np.abs(y_true - y_pred) <= 1.0) * 100.0)
    pct_within_2 = float(np.mean(np.abs(y_true - y_pred) <= 2.0) * 100.0)
    median_ae = float(np.median(np.abs(y_true - y_pred)))
    signed_bias = float(np.mean(y_pred - y_true))
    ci_95 = bootstrap_mae_ci(y_true, y_pred)
    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "pct_within_1": pct_within_1,
        "pct_within_2": pct_within_2,
        "median_ae": median_ae,
        "signed_bias": signed_bias,
        "mae_ci_95": ci_95
    }

def fit_and_evaluate(name, model_warp, model_weft, use_scaled=True):
    X_tr = X_train_scaled if use_scaled else X_train
    X_v = X_val_scaled if use_scaled else X_val
    X_te = X_test_scaled if use_scaled else X_test

    model_warp.fit(X_tr, y_train_warp)
    pred_test_warp = model_warp.predict(X_te)

    model_weft.fit(X_tr, y_train_weft)
    pred_test_weft = model_weft.predict(X_te)

    metrics_w = compute_metrics(y_test_warp, pred_test_warp)
    metrics_wf = compute_metrics(y_test_weft, pred_test_weft)

    return {
        "model": name,
        "pred_warp": pred_test_warp,
        "pred_weft": pred_test_weft,
        "warp": metrics_w,
        "weft": metrics_wf
    }

# 1. XGBoost
xgb_warp = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED)
xgb_weft = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED)
res_xgb = fit_and_evaluate("XGBoost (IFV)", xgb_warp, xgb_weft, use_scaled=False)

# 2. Random Forest
rf_warp = RandomForestRegressor(n_estimators=150, max_depth=8, min_samples_split=3, random_state=SEED)
rf_weft = RandomForestRegressor(n_estimators=150, max_depth=8, min_samples_split=3, random_state=SEED)
res_rf = fit_and_evaluate("RandomForest (IFV)", rf_warp, rf_weft, use_scaled=False)

# 3. Multi-Layer Perceptron (MLP)
mlp_warp = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', max_iter=600, alpha=0.01, random_state=SEED, early_stopping=True)
mlp_weft = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', max_iter=600, alpha=0.01, random_state=SEED, early_stopping=True)
res_mlp = fit_and_evaluate("MLPRegressor (IFV)", mlp_warp, mlp_weft, use_scaled=True)

models_results = [res_xgb, res_rf, res_mlp]

print(f"{'Model':<20} | {'Warp MAE':<9} {'Warp <=1':<9} {'Warp <=2':<9} {'Warp R2':<8} | {'Weft MAE':<9} {'Weft <=1':<9} {'Weft <=2':<9} {'Weft R2':<8}")
print("-" * 100)
for r in models_results:
    print(f"{r['model']:<20} | {r['warp']['mae']:<9.3f} {r['warp']['pct_within_1']:<9.1f} {r['warp']['pct_within_2']:<9.1f} {r['warp']['r2']:<8.3f} | {r['weft']['mae']:<9.3f} {r['weft']['pct_within_1']:<9.1f} {r['weft']['pct_within_2']:<9.1f} {r['weft']['r2']:<8.3f}")


Model                | Warp MAE  Warp <=1  Warp <=2  Warp R2  | Weft MAE  Weft <=1  Weft <=2  Weft R2 
----------------------------------------------------------------------------------------------------
XGBoost (IFV)        | 6.673     19.2      38.5      0.542    | 6.711     7.7       23.1      0.591   
RandomForest (IFV)   | 7.413     7.7       19.2      0.502    | 6.759     3.8       11.5      0.607   
MLPRegressor (IFV)   | 6.831     19.2      34.6      0.462    | 7.071     15.4      26.9      0.600   


In [6]:
# The mechanistic justification for this step is: compare sliding-window IFV models against deterministic baseline, evaluate per-weave performance, and serialize prediction records to CSV and JSON.
with open("results/fabricnet_benchmark.json") as f:
    bench_data = json.load(f)

det_by_num = {s['sample_num']: s for s in bench_data['samples']}
det_warp_preds = np.array([det_by_num[i]['pred_warp'] for i in test_ids], dtype=np.float32)
det_weft_preds = np.array([det_by_num[i]['pred_weft'] for i in test_ids], dtype=np.float32)

metrics_det_w = compute_metrics(y_test_warp, det_warp_preds)
metrics_det_wf = compute_metrics(y_test_weft, det_weft_preds)

res_det = {
    "model": "Deterministic Engine",
    "pred_warp": det_warp_preds,
    "pred_weft": det_weft_preds,
    "warp": metrics_det_w,
    "weft": metrics_det_wf
}

all_evals = models_results + [res_det]

print("\n=== COMPLETE COMPARISON (IFV SLIDING WINDOW) ON 26 HELD-OUT TEST SPECIMENS ===")
print(f"{'Method':<22} | {'Warp MAE':<9} {'Warp <=1':<9} {'Warp <=2':<9} {'Warp R2':<8} | {'Weft MAE':<9} {'Weft <=1':<9} {'Weft <=2':<9} {'Weft R2':<8}")
print("-" * 105)
for r in all_evals:
    print(f"{r['model']:<22} | {r['warp']['mae']:<9.3f} {r['warp']['pct_within_1']:<9.1f} {r['warp']['pct_within_2']:<9.1f} {r['warp']['r2']:<8.3f} | {r['weft']['mae']:<9.3f} {r['weft']['pct_within_1']:<9.1f} {r['weft']['pct_within_2']:<9.1f} {r['weft']['r2']:<8.3f}")

unique_textures = np.unique(test_textures)
weave_breakdown = {}
print("\n=== PER-WEAVE BREAKDOWN (TEST SET) ===")
for tex in unique_textures:
    mask = (test_textures == tex)
    count = int(np.sum(mask))
    print(f"\n--- Texture: {tex.upper()} (N={count}) ---")
    print(f"{'Method':<22} | {'Warp MAE':<9} {'Warp <=2':<9} | {'Weft MAE':<9} {'Weft <=2':<9}")
    weave_breakdown[tex] = {}
    for r in all_evals:
        m_w = float(mean_absolute_error(y_test_warp[mask], r['pred_warp'][mask]))
        p2_w = float(np.mean(np.abs(y_test_warp[mask] - r['pred_warp'][mask]) <= 2.0) * 100.0)
        m_wf = float(mean_absolute_error(y_test_weft[mask], r['pred_weft'][mask]))
        p2_wf = float(np.mean(np.abs(y_test_weft[mask] - r['pred_weft'][mask]) <= 2.0) * 100.0)
        weave_breakdown[tex][r['model']] = {
            "warp_mae": m_w, "warp_pct_within_2": p2_w,
            "weft_mae": m_wf, "weft_pct_within_2": p2_wf
        }
        print(f"{r['model']:<22} | {m_w:<9.3f} {p2_w:<9.1f} | {m_wf:<9.3f} {p2_wf:<9.1f}")

df_preds = pd.DataFrame({
    "sample_id": test_ids,
    "weave_type": test_textures,
    "true_warp": y_test_warp,
    "true_weft": y_test_weft,
    "rf_pred_warp": res_rf['pred_warp'],
    "rf_pred_weft": res_rf['pred_weft'],
    "xgb_pred_warp": res_xgb['pred_warp'],
    "xgb_pred_weft": res_xgb['pred_weft'],
    "mlp_pred_warp": res_mlp['pred_warp'],
    "mlp_pred_weft": res_mlp['pred_weft'],
    "det_pred_warp": det_warp_preds,
    "det_pred_weft": det_weft_preds
})
csv_out = "experiments/learned_models/results/fabricnet_ifv_sliding_window_predictions.csv"
df_preds.to_csv(csv_out, index=False)
print(f"\nPer-sample IFV predictions saved to {csv_out}")

output_summary = {
    "experiment_id": "EXP-FABRICNET-IFV-SLIDING-WINDOW",
    "feature_set": "Published IFV Sliding Window (Secer & Ercan 2023, 160x160 window, stride 80, 4 nested frames)",
    "feature_dimension": int(X_all.shape[1]),
    "test_specimen_count": len(test_ids),
    "test_ids": test_ids,
    "overall_metrics": {r['model']: {"warp": r['warp'], "weft": r['weft']} for r in all_evals},
    "per_weave_breakdown": weave_breakdown
}

json_out = "experiments/learned_models/results/fabricnet_ifv_sliding_window_results.json"
with open(json_out, "w") as f:
    json.dump(output_summary, f, indent=2)

print(f"Results successfully saved to {json_out}")



=== COMPLETE COMPARISON (IFV SLIDING WINDOW) ON 26 HELD-OUT TEST SPECIMENS ===
Method                 | Warp MAE  Warp <=1  Warp <=2  Warp R2  | Weft MAE  Weft <=1  Weft <=2  Weft R2 
---------------------------------------------------------------------------------------------------------
XGBoost (IFV)          | 6.673     19.2      38.5      0.542    | 6.711     7.7       23.1      0.591   
RandomForest (IFV)     | 7.413     7.7       19.2      0.502    | 6.759     3.8       11.5      0.607   
MLPRegressor (IFV)     | 6.831     19.2      34.6      0.462    | 7.071     15.4      26.9      0.600   
Deterministic Engine   | 13.271    19.2      30.8      -0.861   | 13.162    19.2      23.1      -0.397  

=== PER-WEAVE BREAKDOWN (TEST SET) ===

--- Texture: PLAIN (N=10) ---
Method                 | Warp MAE  Warp <=2  | Weft MAE  Weft <=2 
XGBoost (IFV)          | 5.191     40.0      | 6.207     10.0     
RandomForest (IFV)     | 6.162     10.0      | 7.054     10.0     
MLPRegressor (IFV